In [4]:
import pandas as pd
import numpy as np

In [5]:
# Load cleaned dataset
df = pd.read_csv("../data/processed/telco_churn_cleaned.csv")
df.shape

(7032, 21)

In [6]:
# Drop customerID - not a predictive feature
df.drop(columns=['customerID'], inplace=True)
df.shape

(7032, 20)

In [8]:
# Check categorical columns
cat_cols = df.select_dtypes(include='str').columns.tolist()
print(cat_cols)

['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [9]:
# Check unique values per categorical column
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

gender: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Partner: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Dependents: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
PhoneService: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines: <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
OnlineSecurity: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup: <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingMovies: <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Contract: <StringArray>
['Month-to-month', 'One year', '

In [10]:
# Step 1: Label encode binary Yes/No columns
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
binary_map = {'Yes': 1, 'No': 0, 'Female': 0, 'Male': 1}

for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Step 2: Simplify 'No internet/phone service' to 'No', then label encode
service_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_cols:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Step 3: One-hot encode nominal multi-class columns
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=False)

print(df.shape)
print(df.dtypes)

(7032, 27)
gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn                                        int64
InternetService_DSL                           bool
InternetService_Fiber optic                   bool
InternetService_No  

In [11]:
# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)
print(df.dtypes)

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
PaperlessBilling                             int64
MonthlyCharges                             float64
TotalCharges                               float64
Churn                                        int64
InternetService_DSL                          int64
InternetService_Fiber optic                  int64
InternetService_No             

In [12]:
# Feature engineering: create tenure group and drop TotalCharges (multicollinear with tenure)
# TotalCharges = tenure * MonthlyCharges approximately, so it adds no new information

df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)  # +1 to avoid division by zero
df.drop(columns=['TotalCharges'], inplace=True)

print(df.shape)

(7032, 27)


In [13]:
# Split features and target
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

X = df.drop(columns=['Churn'])
y = df['Churn']

# Train-test split before SMOTE (important: SMOTE only on training data, never on test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")

# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}")

Before SMOTE: {0: 4130, 1: 1495}
After SMOTE: {0: 4130, 1: 4130}


In [14]:
# Scale numerical features - important for logistic regression baseline
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'AvgMonthlySpend']

X_train_sm[num_cols] = scaler.fit_transform(X_train_sm[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_sm[num_cols].describe().round(2))

        tenure  MonthlyCharges  AvgMonthlySpend
count  8260.00         8260.00          8260.00
mean     -0.00           -0.00             0.00
std       1.00            1.00             1.00
min      -1.11           -1.73            -1.72
25%      -0.95           -0.78            -0.86
50%      -0.28            0.23             0.10
75%       0.88            0.80             0.84
max       1.84            1.76             1.97


In [15]:
# Save processed data for modeling
import numpy as np

np.save("../data/processed/X_train.npy", X_train_sm)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_train.npy", y_train_sm)
np.save("../data/processed/y_test.npy", y_test)

# Save feature names for later use in SHAP
pd.Series(X_train_sm.columns.tolist()).to_csv("../data/processed/feature_names.csv", index=False)

print("Saved successfully")

Saved successfully
